# Long PDF Summarizer

## 1. Project Overview

**Task:** Summarize long documents that exceed LLM context windows using chunking, map-reduce, and hierarchical strategies — producing both chapter-wise and global summaries.

**Approach:** We simulate a long multi-chapter document, then compare three summarization strategies:
1. **Stuff** — send the whole document (works only for short docs)
2. **Map-reduce** — summarize each chunk independently, then combine
3. **Refine** — iteratively refine a running summary with each chunk

**Stack:**
- `LangChain` + `ChatOllama` — local LLM interaction
- `Ollama` + `qwen3.5:9b` — local LLM (no API keys)

> **No API keys required.** Everything runs locally.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | Handle documents **longer than context windows** |
| 2 | Implement **map-reduce** summarization |
| 3 | Implement **iterative refinement** summarization |
| 4 | Produce **chapter-wise** and **global** summaries |
| 5 | Compare strategies on **quality vs speed** |
| 6 | Understand **information loss** at each stage |

## 3. Problem Statement

Long PDFs (research papers, reports, books) can be 10,000–100,000+ words — far beyond any model's context window. We need strategies that preserve key information while fitting within model constraints.

## 4. Why This Matters

- **Enterprise use case:** legal contracts, annual reports, compliance documents
- **Academic use:** summarize textbooks, survey papers, theses
- **Chunking strategy** is the foundation of almost every RAG and document-processing pipeline

## 5. Environment Setup

In [ ]:
# !pip install -q langchain langchain-ollama langchain-core
print("Dependencies: langchain, langchain-ollama")
print("LLM: Ollama with qwen3.5:9b")

## 6. Imports & Helpers

In [ ]:
import os, json, re, time, warnings
from textwrap import dedent
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

LLM_MODEL = "qwen3.5:9b"
llm = ChatOllama(model=LLM_MODEL, temperature=0)

def clean(text: str) -> str:
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()

def ask(system: str, user: str) -> str:
    resp = llm.invoke([SystemMessage(content=system), HumanMessage(content=user)])
    return clean(resp.content)

test = ask("Reply with one word.", "Say ready.")
print(f"LLM ready: {test[:30]}")

---

## 7. Simulate a Long Document

We create a realistic multi-chapter technical document to test our summarization strategies.

In [ ]:
CHAPTERS = {
    "Chapter 1: Introduction to Cloud Computing": dedent("""Cloud computing has fundamentally transformed how organizations build and deploy technology. Instead of maintaining expensive on-premises data centers, companies can now rent computing resources on demand from providers like AWS, Azure, and Google Cloud. This shift began in the mid-2000s when Amazon launched its Elastic Compute Cloud (EC2) service, allowing developers to provision virtual servers in minutes rather than weeks.

The benefits are significant: reduced capital expenditure, elastic scalability, and global reach. A startup can now access the same infrastructure as a Fortune 500 company for pennies per hour. However, cloud adoption brings challenges including vendor lock-in, data sovereignty concerns, security shared-responsibility models, and cost management complexity. Organizations that fail to monitor their cloud spending often face bill shock, with studies showing 30-35% of cloud spending is wasted on idle or oversized resources.

The cloud market has grown to over $500 billion annually, with AWS commanding roughly 32% market share, Azure at 23%, and Google Cloud at 10%. The remaining market is split among dozens of smaller providers and specialized platforms."""),

    "Chapter 2: Infrastructure as Code": dedent("""Infrastructure as Code (IaC) is the practice of managing and provisioning computing infrastructure through machine-readable definition files rather than physical hardware configuration or interactive configuration tools. The key tools in this space include Terraform, AWS CloudFormation, Pulumi, and Ansible.

Terraform, developed by HashiCorp, has become the de facto standard for multi-cloud IaC. It uses a declarative language called HCL (HashiCorp Configuration Language) to describe the desired state of infrastructure. When executed, Terraform compares the desired state with the current state and makes only the necessary changes. This approach is idempotent, meaning running the same configuration multiple times produces the same result.

Key IaC principles include version control (storing infrastructure definitions in Git), modularity (reusable components), testing (validating configurations before deployment), and drift detection (identifying when actual infrastructure diverges from the defined state). Organizations adopting IaC report 90% fewer configuration errors and 70% faster provisioning times compared to manual processes."""),

    "Chapter 3: Container Orchestration": dedent("""Kubernetes has become the industry standard for container orchestration, managing the deployment, scaling, and operation of containerized applications across clusters of machines. Originally developed by Google and released as open source in 2014, Kubernetes (often abbreviated as K8s) draws on Google's 15 years of experience running production workloads at scale.

A Kubernetes cluster consists of a control plane (which manages the cluster) and worker nodes (which run the actual workloads). The control plane includes the API server, scheduler, controller manager, and etcd (a distributed key-value store for cluster state). Worker nodes run the kubelet agent, container runtime (like containerd), and kube-proxy for networking.

Key Kubernetes concepts include Pods (the smallest deployable unit), Deployments (declarative updates for Pods), Services (stable network endpoints), ConfigMaps and Secrets (configuration management), and Horizontal Pod Autoscaling (automatic scaling based on metrics). The ecosystem includes tools like Helm for package management, Istio for service mesh, and ArgoCD for GitOps-based continuous deployment."""),

    "Chapter 4: Observability and Monitoring": dedent("""Modern distributed systems require comprehensive observability to understand system behavior, diagnose issues, and maintain reliability. The three pillars of observability are metrics (numerical measurements over time), logs (discrete event records), and traces (request paths through distributed systems).

Prometheus has become the standard for metrics collection in cloud-native environments. It uses a pull-based model, scraping metrics endpoints at regular intervals, and stores data in a time-series database. Grafana provides visualization dashboards for Prometheus metrics and other data sources.

Distributed tracing tools like Jaeger and Zipkin allow engineers to follow a single request as it traverses multiple microservices. This is essential for diagnosing latency issues and understanding dependencies. OpenTelemetry has emerged as the unified standard for instrumentation, providing vendor-neutral APIs for metrics, logs, and traces.

Effective alerting requires careful threshold design. Alert fatigue — where teams receive too many non-actionable alerts — is a common problem. Best practices include setting alerts on symptoms rather than causes, using multi-signal alerts, and implementing escalation policies with clear ownership."""),

    "Chapter 5: Security and Compliance": dedent("""Cloud security operates on a shared responsibility model: the cloud provider secures the infrastructure, while the customer secures their data, applications, and access management. This distinction is critical — many security breaches occur because organizations assume the cloud provider handles everything.

Identity and Access Management (IAM) is the foundation of cloud security. The principle of least privilege dictates that users and services should have only the minimum permissions necessary. Role-based access control (RBAC) groups permissions into roles that can be assigned to users, while attribute-based access control (ABAC) makes decisions based on attributes of the user, resource, and environment.

Data encryption should be applied both at rest and in transit. At rest, cloud providers offer server-side encryption with provider-managed keys, customer-managed keys (BYOK), or hardware security modules (HSMs). In transit, TLS 1.3 is the current standard for securing network communications.

Compliance frameworks like SOC 2, ISO 27001, HIPAA, and GDPR impose specific requirements on data handling, access controls, and audit logging. Automated compliance scanning tools like AWS Config, Azure Policy, and open-source tools like Checkov can continuously verify that infrastructure meets compliance requirements."""),
}

full_doc = "\n\n".join(f"{title}\n{body}" for title, body in CHAPTERS.items())
print(f"Document: {len(CHAPTERS)} chapters, {len(full_doc):,d} characters, ~{len(full_doc.split()):,d} words")
for title in CHAPTERS:
    print(f"  {title}: {len(CHAPTERS[title]):,d} chars")

---

## 8. Strategy 1: Chapter-wise Summaries (Map)

Summarize each chapter independently. This is the "map" phase of map-reduce.

In [ ]:
CHAPTER_SYSTEM = """You summarize technical document chapters.
Rules:
- Write 2-3 sentences capturing the key concepts
- Include specific numbers, tools, or standards mentioned
- Be precise and factual
- Return ONLY the summary text"""

chapter_summaries = {}
t0 = time.perf_counter()

for title, body in CHAPTERS.items():
    summary = ask(CHAPTER_SYSTEM, f"Chapter title: {title}\n\nContent:\n{body}\n\nSummary:")
    chapter_summaries[title] = summary
    print(f"\n{title}")
    print(f"  {summary}")

map_time = time.perf_counter() - t0
print(f"\nTotal map time: {map_time:.1f}s")

## 9. Strategy 2: Map-Reduce — Global Summary

Take the chapter summaries and combine them into a single global summary (the "reduce" phase).

In [ ]:
REDUCE_SYSTEM = """You combine chapter summaries into one cohesive document summary.
Rules:
- Write 4-6 sentences covering the entire document
- Capture the overall narrative arc, not just individual chapters
- Mention the most important tools, numbers, and concepts
- Return ONLY the summary"""

combined = "\n\n".join(f"{title}:\n{summary}" for title, summary in chapter_summaries.items())

t0 = time.perf_counter()
global_summary_mr = ask(REDUCE_SYSTEM, f"Chapter summaries:\n{combined}\n\nGlobal summary:")
reduce_time = time.perf_counter() - t0

print("MAP-REDUCE GLOBAL SUMMARY")
print("=" * 50)
print(global_summary_mr)
print(f"\nReduce time: {reduce_time:.1f}s | Total: {map_time + reduce_time:.1f}s")

## 10. Strategy 3: Iterative Refinement

Start with a summary of the first chunk, then refine it with each subsequent chunk. This preserves context better than map-reduce but is sequential (slower).

In [ ]:
REFINE_SYSTEM = """You are refining a document summary with new information from the next chapter.
Rules:
- Start from the existing summary and integrate new information
- Keep the total summary to 4-6 sentences
- Drop less important details to make room for new ones
- Maintain a coherent narrative
- Return ONLY the refined summary"""

chapter_list = list(CHAPTERS.items())
t0 = time.perf_counter()

# Start with first chapter
running_summary = ask(CHAPTER_SYSTEM, f"Chapter: {chapter_list[0][0]}\n\n{chapter_list[0][1]}\n\nSummary:")
print(f"After Chapter 1: {running_summary[:100]}...")

# Refine with each subsequent chapter
for title, body in chapter_list[1:]:
    running_summary = ask(
        REFINE_SYSTEM,
        f"Existing summary:\n{running_summary}\n\nNew chapter: {title}\n{body}\n\nRefined summary:"
    )
    chapter_num = title.split(":")[0]
    print(f"After {chapter_num}: {running_summary[:100]}...")

refine_time = time.perf_counter() - t0

print(f"\nREFINE GLOBAL SUMMARY")
print("=" * 50)
print(running_summary)
print(f"\nRefine time: {refine_time:.1f}s")

## 11. Compare the Two Strategies

In [ ]:
print("STRATEGY COMPARISON")
print("=" * 60)

print(f"\nMap-Reduce ({map_time + reduce_time:.1f}s):")
print(f"  {global_summary_mr}")
print(f"  Words: {len(global_summary_mr.split())}")

print(f"\nIterative Refine ({refine_time:.1f}s):")
print(f"  {running_summary}")
print(f"  Words: {len(running_summary.split())}")

print("\nTradeoffs:")
print("  Map-Reduce: parallel-friendly, may lose cross-chapter connections")
print("  Refine: preserves narrative flow, sequential (slower), may bias toward later chapters")

## 12. Multi-Level Output — Executive, Detailed, and Chapter-Level

In [ ]:
EXEC_SYSTEM = """Write a 2-sentence executive summary of this document. Focus on the single most important takeaway and one supporting detail."""

exec_summary = ask(EXEC_SYSTEM, f"Chapter summaries:\n{combined}\n\nExecutive summary:")

print("MULTI-LEVEL SUMMARIES")
print("=" * 60)

print("\n\u2501 EXECUTIVE (2 sentences):")
print(f"  {exec_summary}")

print("\n\u2501 GLOBAL (4-6 sentences):")
print(f"  {global_summary_mr}")

print("\n\u2501 CHAPTER-LEVEL:")
for title, summary in chapter_summaries.items():
    print(f"  {title}")
    print(f"    {summary}\n")

## 13. Key Concept Extraction

In [ ]:
CONCEPTS_SYSTEM = """Extract the 10 most important technical concepts from this document.
For each, provide: name, one-sentence definition, and which chapter it appears in.
Return as a numbered list."""

concepts = ask(CONCEPTS_SYSTEM, f"Document content:\n{combined}\n\nKey concepts:")
print("KEY CONCEPTS")
print("=" * 50)
print(concepts)

---

## 14. Error Analysis

In [ ]:
# Test: Does map-reduce lose cross-chapter connections?
CROSS_REF_SYSTEM = """Answer this question using ONLY the document content provided."""

question = "How does observability (Chapter 4) relate to security compliance (Chapter 5)?"

# From chapter summaries only (simulating map-reduce context)
answer_mr = ask(CROSS_REF_SYSTEM, f"Document summaries:\n{combined}\n\nQuestion: {question}")

# From full document
answer_full = ask(CROSS_REF_SYSTEM, f"Document:\n{full_doc}\n\nQuestion: {question}")

print("CROSS-CHAPTER QUESTION TEST")
print("=" * 50)
print(f"Q: {question}")
print(f"\nFrom summaries: {answer_mr[:200]}")
print(f"\nFrom full doc:  {answer_full[:200]}")
print("\n\u2192 Map-reduce summaries often lose cross-chapter connections")

## 15. Common Mistakes

| Mistake | Fix |
|---------|-----|
| Sending full document to LLM | Use chunking strategies |
| Equal-length chunks ignoring structure | Chunk by chapters/sections |
| Map-reduce with no deduplication | Merge step should remove redundancy |
| Refine that grows unbounded | Set max summary length at each step |
| No multi-level output | Provide executive + detailed + chapter summaries |

## 16. Production Ideas

1. **PDF parsing** — use `pymupdf` or `pdfplumber` to extract text with section headers
2. **Table extraction** — handle tables separately from prose
3. **Figure captions** — include figure descriptions in the summary
4. **Citation tracking** — note which sections support each summary claim
5. **Incremental updates** — when a document is revised, only re-summarize changed sections

## 17. Exercises

In [ ]:
# Exercise: Generate a Q&A study guide from the document
QA_SYSTEM = """Generate 5 study questions from this document chapter summary.
Each question should test understanding of a key concept.
Return as a numbered list."""

for title, summary in list(chapter_summaries.items())[:2]:
    questions = ask(QA_SYSTEM, f"{title}:\n{summary}\n\nStudy questions:")
    print(f"\n{title}")
    print(questions)

## 18. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Chunk by structure** (chapters/sections), not by arbitrary character count |
| 2 | **Map-reduce** is faster and parallelizable but may lose cross-chunk connections |
| 3 | **Iterative refine** preserves narrative flow but is sequential and may bias toward later content |
| 4 | **Multi-level summaries** (executive, detailed, chapter) serve different audiences |
| 5 | **Key concept extraction** complements summaries with structured knowledge |
| 6 | Always **test cross-chapter questions** to check for information loss |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*